**Axon Merging - Suite2p Pipeline**

Merges correlated axonal ROIs from suite2p output by grouping and averaging their dF/F traces.  
Assumes ROI identification/filtering (`Axon_Identification_Merge_Suite2p.ipynb`) has already been run  
and `iscell.npy` reflects the filtered ROIs.

In [ ]:
import numpy as np
import matplotlib.pyplot as pltz
import scipy.stats
import os
import itertools
from collections import defaultdict

%cd "C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation"
from helper import axons
from helper.correlation import corr2_coeff
from helper.twop import TwoP
from helper.files import write_h5



# ============================================================================
# CONFIGURATION
# ============================================================================
suite2p_path = r"D:\V1_SpatialModulation\2p\V1_axonal\JSY060_ChronicImaging_prism\260303_JSY_JSY060_LongitudinalImaging_Axonal_Prism_Day7\TSeries-03032026-0817-001\suite2p\plane0"
save_path = os.path.join(suite2p_path, 'axon_merged_output.h5')
fig_save_path = os.path.join(suite2p_path, 'merging_figures')
os.makedirs(fig_save_path, exist_ok=True)

twop_rate = 10.047       # imaging frame rate (Hz)
neu_correction = 0     # neuropil correction factor
cc_thresh = 0.5          # pairwise correlation threshold for merging
max_distance = 30        # max centroid distance (px) for a direct edge — only used if use_max_distance=True
use_max_distance = True  # True  → local edges (cc AND dist); long-range chaining still allowed
                         # False → correlation-only edges; no spatial constraint at all
merge_duplicates = True  # True = average correlated groups, False = drop weaker of pair

In [ ]:
# ============================================================================
# LOAD SUITE2P DATA AND COMPUTE dF/F
# ============================================================================

# Load suite2p files
F = np.load(os.path.join(suite2p_path, 'F.npy'))
Fneu = np.load(os.path.join(suite2p_path, 'Fneu.npy'))
iscell = np.load(os.path.join(suite2p_path, 'iscell.npy'))
stat = np.load(os.path.join(suite2p_path, 'stat.npy'), allow_pickle=True)
ops = np.load(os.path.join(suite2p_path, 'ops.npy'), allow_pickle=True).item()

# Filter by iscell (assumes ROI identification has already been run)
cell_mask = iscell[:, 0] == 1
F_cells = F[cell_mask, :]
Fneu_cells = Fneu[cell_mask, :]
stat_cells = stat[cell_mask]
original_indices = np.where(cell_mask)[0]  # maps filtered idx -> original ROI idx

print(f"Total ROIs: {len(stat)}")
print(f"Cells after iscell filter: {F_cells.shape[0]}")
print(f"Frames: {F_cells.shape[1]}")

# Compute dF/F with neuropil correction (same method as helper/twop.py TwoP.calc_dFF)
nCells, lenT = F_cells.shape
dFF = np.zeros((nCells, lenT))

for c in range(nCells):
    F_cell = F_cells[c, :].copy()
    F_neu = Fneu_cells[c, :].copy()

    # Neuropil subtraction
    norm_F = F_cell - neu_correction * F_neu + neu_correction * np.nanmean(F_neu)

    # Baseline estimation (mode)
    F0 = scipy.stats.mode(norm_F, nan_policy='omit').mode

    # dF/F (%)
    dFF[c, :] = (norm_F - F0) / F0 * 100

# Mean fluorescence across all ROIs per frame (proxy for global frame fluorescence)
frame_means = np.mean(F_cells, axis=0)

print(f"\ndF/F computed: {dFF.shape}")

In [ ]:
# ============================================================================
# COMPUTE PAIRWISE CORRELATIONS AND DISTANCES
# ============================================================================

# Get ROI centroids (used for spatial visualisation and distance constraint)
centroids = np.array([s['med'] for s in stat_cells])  # (nCells, 2) = (y, x)

# Full correlation matrix — vectorised
corr_mat = np.corrcoef(dFF)  # (nCells, nCells)

# Flatten upper-triangle to vectors indexed by pair
perm_mat = np.array(list(itertools.combinations(range(nCells), 2)))
cc_vec   = corr_mat[perm_mat[:, 0], perm_mat[:, 1]]

# Euclidean centroid distances for every pair (same indexing as cc_vec)
diff     = centroids[perm_mat[:, 0]] - centroids[perm_mat[:, 1]]
dist_vec = np.sqrt((diff ** 2).sum(axis=1))

print(f"Pairwise correlations computed: {len(cc_vec)} pairs")
print(f"Pairwise distances computed:    {len(dist_vec)} pairs")

In [ ]:
# import matplotlib.pyplot as plt
# # ============================================================================
# # PARAMETER SWEEP: neu_correction × cc_thresh × max_distance
# # For each combination: print summary + save spatial map (after merging only)
# # ============================================================================

# neu_vals  = [0.0, 0.1, 0.2, 0.3, 0.7]
# cc_vals   = [0.5, 0.6, 0.7]
# dist_vals = [15, 20, 25, 30, 40, 50]

# mean_img = ops.get('meanImg', np.zeros((ops['Ly'], ops['Lx'])))
# vmin_img = np.percentile(mean_img, 1)
# vmax_img = np.percentile(mean_img, 99)
# cmap_groups = plt.cm.tab10

# # Summary table header
# print(f"{'neu':>5} {'cc':>5} {'dist':>5} | {'n_pairs':>8} {'largest':>8} {'n_final':>8} {'n_multi':>8}")
# print("-" * 65)

# for neu in neu_vals:

#     # Recompute dF/F for this neu_correction
#     dff_s = np.zeros((nCells, lenT))
#     for c in range(nCells):
#         F_c   = F_cells[c, :].copy()
#         F_neu = Fneu_cells[c, :].copy()
#         norm_F = F_c - neu * F_neu + neu * np.nanmean(F_neu)
#         F0 = scipy.stats.mode(norm_F, nan_policy='omit').mode
#         dff_s[c, :] = (norm_F - F0) / F0 * 100

#     # Pairwise correlations for this neu (reuse perm_mat, dist_vec from main notebook)
#     corr_mat_s = np.corrcoef(dff_s)
#     cc_vec_s   = corr_mat_s[perm_mat[:, 0], perm_mat[:, 1]]

#     for cc in cc_vals:
#         for max_dist in dist_vals:

#             # --- Build edges ---
#             dist_mask_s = dist_vec < max_dist
#             edge_mask_s = (cc_vec_s > cc) & dist_mask_s
#             n_pairs     = int(np.sum(edge_mask_s))

#             # --- Connected components ---
#             adj = defaultdict(set)
#             for idx in np.where(edge_mask_s)[0]:
#                 a, b = perm_mat[idx]
#                 adj[a].add(b)
#                 adj[b].add(a)

#             visited_s, groups_s = set(), []
#             for node in range(nCells):
#                 if node not in visited_s:
#                     stack, grp = [node], set()
#                     while stack:
#                         n = stack.pop()
#                         if n not in visited_s:
#                             visited_s.add(n)
#                             grp.add(n)
#                             stack.extend(adj[n] - visited_s)
#                     groups_s.append(sorted(grp))

#             n_final   = len(groups_s)
#             largest   = max(len(g) for g in groups_s)
#             n_multi   = sum(1 for g in groups_s if len(g) > 1)

#             print(f"{neu:>5.1f} {cc:>5.2f} {max_dist:>5d} | "
#                   f"{n_pairs:>8d} {largest:>8d} {n_final:>8d} {n_multi:>8d}")

#             # --- Spatial map (after merging only) ---
#             fig, ax = plt.subplots(1, 1, figsize=(8, 7), dpi=120)

#             ax.imshow(mean_img, cmap='gray', vmin=vmin_img, vmax=vmax_img)

#             # Single ROI groups → gray
#             for g in groups_s:
#                 if len(g) == 1:
#                     ax.plot(stat_cells[g[0]]['xpix'], stat_cells[g[0]]['ypix'],
#                             '.', color='gray', ms=0.3, alpha=0.4)

#             # Multi-ROI groups → color, sized by group size for visibility
#             color_idx = 0
#             for g in sorted(groups_s, key=len):   # draw largest on top
#                 if len(g) > 1:
#                     color = cmap_groups(color_idx % 10)
#                     color_idx += 1
#                     for gx in g:
#                         ax.plot(stat_cells[gx]['xpix'], stat_cells[gx]['ypix'],
#                                 '.', color=color, ms=0.5, alpha=0.8)

#             ax.set_title(
#                 f'After merging: {n_final} axons  |  largest={largest}  multi={n_multi}\n'
#                 f'neu={neu:.1f}  |  cc>{cc:.2f}  |  max_dist={max_dist}px',
#                 fontsize=10)
#             ax.axis('off')

#             plt.tight_layout()

#             fname = f'sweep_neu{int(neu*10):02d}_cc{int(cc*10):02d}_dist{max_dist:02d}.png'
#             plt.savefig(os.path.join(fig_save_path, fname),
#                         bbox_inches='tight', dpi=120)
#             plt.close()   # close instead of show to avoid flooding notebook

# print("\nAll figures saved to:", fig_save_path)


In [ ]:
# import matplotlib.pyplot as plt
# # ============================================================================
# # PARAMETER SWEEP: neu_correction × cc_thresh
# # For each combination: print summary + save spatial map (before/after)
# # ============================================================================

# neu_vals  = [0.0, 0.1, 0.2, 0.3, 0.7]
# cc_vals   = [0.5, 0.6, 0.7]
# dist_vals = [None]  # no distance constraint

# mean_img = ops.get('meanImg', np.zeros((ops['Ly'], ops['Lx'])))
# vmin_img = np.percentile(mean_img, 1)
# vmax_img = np.percentile(mean_img, 99)
# cmap_groups = plt.cm.tab10

# # Summary table header
# print(f"{'neu':>5} {'cc':>5} {'dist':>5} | {'n_pairs':>8} {'largest':>8} {'n_final':>8} {'n_multi':>8}")
# print("-" * 65)

# for neu in neu_vals:

#     # Recompute dF/F for this neu_correction
#     dff_s = np.zeros((nCells, lenT))
#     for c in range(nCells):
#         F_c   = F_cells[c, :].copy()
#         F_neu = Fneu_cells[c, :].copy()
#         norm_F = F_c - neu * F_neu + neu * np.nanmean(F_neu)
#         F0 = scipy.stats.mode(norm_F, nan_policy='omit').mode
#         dff_s[c, :] = (norm_F - F0) / F0 * 100

#     # Pairwise correlations for this neu (reuse perm_mat, dist_vec from main notebook)
#     corr_mat_s = np.corrcoef(dff_s)
#     cc_vec_s   = corr_mat_s[perm_mat[:, 0], perm_mat[:, 1]]

#     for cc in cc_vals:
#         for max_dist in dist_vals:

#             # --- Build edges ---
#             edge_mask_s = cc_vec_s > cc
#             n_pairs     = int(np.sum(edge_mask_s))

#             # --- Connected components ---
#             adj = defaultdict(set)
#             for idx in np.where(edge_mask_s)[0]:
#                 a, b = perm_mat[idx]
#                 adj[a].add(b)
#                 adj[b].add(a)

#             visited_s, groups_s = set(), []
#             for node in range(nCells):
#                 if node not in visited_s:
#                     stack, grp = [node], set()
#                     while stack:
#                         n = stack.pop()
#                         if n not in visited_s:
#                             visited_s.add(n)
#                             grp.add(n)
#                             stack.extend(adj[n] - visited_s)
#                     groups_s.append(sorted(grp))

#             n_final   = len(groups_s)
#             largest   = max(len(g) for g in groups_s)
#             n_multi   = sum(1 for g in groups_s if len(g) > 1)

#             # print(f"{neu:>5.1f} {cc:>5.2f} {max_dist:>5d} | "
#             #       f"{n_pairs:>8d} {largest:>8d} {n_final:>8d} {n_multi:>8d}")

#             # --- Spatial map ---
#             fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=120)

#             # Left: before merging (same for all, but included for reference)
#             ax = axes[0]
#             ax.imshow(mean_img, cmap='gray', vmin=vmin_img, vmax=vmax_img)
#             for i in range(len(stat_cells)):
#                 ax.plot(stat_cells[i]['xpix'], stat_cells[i]['ypix'],
#                         'c.', ms=0.3, alpha=0.5)
#             ax.set_title(f'Before merging: {nCells} ROIs', fontsize=10)
#             ax.axis('off')

#             # Right: after merging
#             ax = axes[1]
#             ax.imshow(mean_img, cmap='gray', vmin=vmin_img, vmax=vmax_img)

#             # Single ROI groups → gray
#             for g in groups_s:
#                 if len(g) == 1:
#                     ax.plot(stat_cells[g[0]]['xpix'], stat_cells[g[0]]['ypix'],
#                             '.', color='gray', ms=0.3, alpha=0.4)

#             # Multi-ROI groups → color, sized by group size for visibility
#             color_idx = 0
#             for g in sorted(groups_s, key=len):   # draw largest on top
#                 if len(g) > 1:
#                     color = cmap_groups(color_idx % 10)
#                     color_idx += 1
#                     for gx in g:
#                         ax.plot(stat_cells[gx]['xpix'], stat_cells[gx]['ypix'],
#                                 '.', color=color, ms=0.5, alpha=0.8)

#             ax.set_title(
#                 f'After merging: {n_final} axons  |  largest={largest}  multi={n_multi}',
#                 fontsize=10)
#             ax.axis('off')

#             fig.suptitle(
#                 f'neu={neu:.1f}  |  cc>{cc:.2f}  |  no distance constraint',
#                 fontsize=12, fontweight='bold')
#             plt.tight_layout()
            
#             fname = f'sweep_neu{int(neu*10):02d}_cc{int(cc*10):02d}.png'
#             plt.savefig(os.path.join(fig_save_path, fname),
#                         bbox_inches='tight', dpi=120)
#             plt.close()   # close instead of show to avoid flooding notebook

# print("\nAll figures saved to:", fig_save_path)

In [ ]:
# ============================================================================
# RUN AXON MERGING — local cc + distance edges, global chaining
# ============================================================================
# Edge rule: ROI_A → ROI_B only if BOTH:
#   (1) corr(A, B) > cc_thresh          — temporally similar
#   (2) distance(A, B) < max_distance   — neighbouring boutons on same axon
#                                         (only applied if use_max_distance=True)
#
# Long-range merging (A ←→ D spanning the FOV) still happens automatically
# through connected-component chaining:
#   A ──d<max── B ──d<max── C ──d<max── D
# No post-hoc splitting — chaining is the intended behaviour.

dist_mask = (dist_vec < max_distance) if use_max_distance else np.ones(len(cc_vec), dtype=bool)
edge_mask  = (cc_vec > cc_thresh) & dist_mask

adjacency = defaultdict(set)
for idx in np.where(edge_mask)[0]:
    a, b = perm_mat[idx]
    adjacency[a].add(b)
    adjacency[b].add(a)

n_corr_only = int(np.sum(cc_vec > cc_thresh))
n_edges     = int(np.sum(edge_mask))
if use_max_distance:
    print(f"Spatial constraint: ON  (max_distance={max_distance} px)")
    print(f"Pairs above cc_thresh ({cc_thresh}):           {n_corr_only}")
    print(f"Pairs also within max_distance:             {n_edges}")
    print(f"Pairs removed by distance filter:           {n_corr_only - n_edges}")
else:
    print(f"Spatial constraint: OFF")
    print(f"Pairs above cc_thresh ({cc_thresh}):           {n_edges}")

# Connected components (chaining gives long-range merging for free)
visited = set()
kept_groups = []
for node in range(nCells):
    if node not in visited:
        stack = [node]
        group = set()
        while stack:
            n = stack.pop()
            if n not in visited:
                visited.add(n)
                group.add(n)
                stack.extend(adjacency[n] - visited)
        kept_groups.append(sorted(list(group)))

# Average traces within each group
dFF_merged = np.array([np.mean(dFF[g, :], axis=0) for g in kept_groups])

# Denoise and infer spikes
denoised_dFF, spikes = TwoP.calc_inf_spikes(dFF_merged, fps=twop_rate)

n_original     = nCells
n_merged       = dFF_merged.shape[0]
n_multi_groups = sum(1 for g in kept_groups if len(g) > 1)

print(f"\nOriginal axons:             {n_original}")
print(f"After merging:              {n_merged} independent axons")
print(f"Groups with >1 axon merged: {n_multi_groups}")
print(f"Largest merged group:       {max(len(g) for g in kept_groups)} ROIs")

In [ ]:
import matplotlib.pyplot as plt
# ============================================================================
# VISUALIZE MERGED GROUPS ON MEAN IMAGE
# ============================================================================

mean_img = ops.get('meanImg', np.zeros((ops['Ly'], ops['Lx'])))

fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=150)

# Left: all ROIs before merging
ax = axes[0]
ax.imshow(mean_img, cmap='gray',
          vmin=np.percentile(mean_img, 1), vmax=np.percentile(mean_img, 99))
for i in range(len(stat_cells)):
    ypix = stat_cells[i]['ypix']
    xpix = stat_cells[i]['xpix']
    ax.plot(xpix, ypix, 'c.', ms=0.3, alpha=0.5)
ax.set_title(f'Before merging: {n_original} ROIs')
ax.axis('off')

# Right: after merging - single groups in gray, multi groups in color
ax = axes[1]
ax.imshow(mean_img, cmap='gray',
          vmin=np.percentile(mean_img, 1), vmax=np.percentile(mean_img, 99))

# Single-axon groups in gray
for gi, g in enumerate(kept_groups):
    if len(g) == 1:
        idx = g[0] if isinstance(g, list) else g
        ypix = stat_cells[idx]['ypix']
        xpix = stat_cells[idx]['xpix']
        ax.plot(xpix, ypix, '.', color='gray', ms=0.3, alpha=0.5)

# Multi-axon groups in distinct colors
cmap = plt.cm.tab10
color_idx = 0
for gi, g in enumerate(kept_groups):
    if len(g) > 1:
        color = cmap(color_idx % 10)
        color_idx += 1
        for gx in g:
            ypix = stat_cells[gx]['ypix']
            xpix = stat_cells[gx]['xpix']
            ax.plot(xpix, ypix, '.', color=color, ms=0.5, alpha=0.8)

ax.set_title(f'After merging: {n_merged} independent axons')
ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(fig_save_path, 'roi_before_after_merging.png'), bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ============================================================================
# PLOT TRACES OF MERGED GROUPS
# ============================================================================

multi_groups = [(gi, g) for gi, g in enumerate(kept_groups) if len(g) > 1]
n_show = len(multi_groups)
# n_show = min(10, len(multi_groups))

if n_show > 0:
    fig, axes = plt.subplots(n_show, 1, figsize=(12, 2 * n_show), dpi=150)
    if n_show == 1:
        axes = [axes]

    # Show first 100 seconds
    n_frames_show = min(int(100 * twop_rate), lenT)
    time = np.arange(n_frames_show) / twop_rate

    for i, (gi, g) in enumerate(multi_groups[:n_show]):
        ax = axes[i]
        # Individual traces (transparent)
        for gx in g:
            ax.plot(time, dFF[gx, :n_frames_show], alpha=0.4, lw=0.5)
        # Merged trace (black)
        ax.plot(time, dFF_merged[gi, :n_frames_show], 'k-', lw=1, label='merged')
        ax.set_ylabel('dF/F (%)')
        ax.set_title(f'Group {gi}: {len(g)} axons merged (ROIs {g})', fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if i == 0:
            ax.legend(fontsize=8)

    axes[-1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.savefig(os.path.join(fig_save_path, 'merged_group_traces.png'), bbox_inches='tight', dpi=150)
    plt.show()
else:
    print("No multi-axon groups found with current thresholds.")

In [ ]:
# ============================================================================
# VERIFY MERGED GROUPS: 4x2 grand figures (8 groups per figure)
# ============================================================================

multi_groups = [(gi, g) for gi, g in enumerate(kept_groups) if len(g) > 1]
n_rows, n_cols = 4, 4
groups_per_fig = n_rows * n_cols
n_figs = int(np.ceil(len(multi_groups) / groups_per_fig))

n_frames_show = min(int(100 * twop_rate), lenT)
time = np.arange(n_frames_show) / twop_rate

for fig_i in range(n_figs):
    start = fig_i * groups_per_fig
    end = min(start + groups_per_fig, len(multi_groups))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 12), dpi=100)
    axes_flat = axes.flatten()

    for j, ax in enumerate(axes_flat):
        idx = start + j
        if idx < end:
            gi, g = multi_groups[idx]
            # Individual traces
            for gx in g:
                ax.plot(time, dFF[gx, :n_frames_show], alpha=0.4, lw=0.5)
            # Merged trace
            ax.plot(time, dFF_merged[gi, :n_frames_show], 'k-', lw=1)
            ax.set_title(f'Group {gi}: {len(g)} ROIs {g}', fontsize=7)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(labelsize=6)
        else:
            ax.axis('off')

    fig.suptitle(f'Merged groups {start+1}–{end} of {len(multi_groups)}', fontsize=11)
    fig.supxlabel('Time (s)', fontsize=9)
    fig.supylabel('dF/F (%)', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_save_path, f'merged_groups_fig{fig_i+1:02d}.png'), bbox_inches='tight', dpi=100)
    plt.show()

print(f"Plotted {len(multi_groups)} merged groups across {n_figs} figures")

In [ ]:
# For each multi-axon group, keep the ROI with highest dF/F kurtosis as the
# representative (most event-driven, best signal quality) and set the others to iscell=0.
iscell_updated = iscell.copy()
n_removed = 0

for g in kept_groups:
    if len(g) > 1:
        # Find the ROI with highest Fisher kurtosis of dF/F in this group
        kurt_vals = [scipy.stats.kurtosis(dFF[idx, :], fisher=True) for idx in g]
        best_idx = g[np.argmax(kurt_vals)]
        # Set all others in this group to iscell=0
        for idx in g:
            if idx != best_idx:
                orig_idx = original_indices[idx]
                iscell_updated[orig_idx, 0] = 0
                n_removed += 1

n_before = int(np.sum(iscell[:, 0] == 1))
n_after = int(np.sum(iscell_updated[:, 0] == 1))

print(f"\nOriginal ROIs: {n_before}")
print(f"Updated ROIs:  {n_after}")
print(f"Removed ROIs:  {n_removed}")

In [ ]:
# ============================================================================
# UPDATE ISCELL.NPY
# ============================================================================
# For each merged group, keep the ROI with highest dF/F kurtosis as the
# representative (most event-driven, best signal quality) and set the others
# to iscell=0. Single-axon groups are unchanged.
# Downstream pipeline reads F.npy[iscell==1] as usual — one row per axon, no oversampling.

iscell_premerge_path = os.path.join(suite2p_path, 'iscell_premerge.npy')
iscell_path = os.path.join(suite2p_path, 'iscell.npy')

# Backup
if not os.path.exists(iscell_premerge_path):
    np.save(iscell_premerge_path, iscell)
    print(f"Backup saved: {iscell_premerge_path}")
else:
    print(f"Backup already exists: {iscell_premerge_path}")

# For each multi-axon group, keep the ROI with highest Fisher kurtosis of dF/F
iscell_updated = iscell.copy()
n_removed = 0

for g in kept_groups:
    if len(g) > 1:
        kurt_vals = [scipy.stats.kurtosis(dFF[idx, :], fisher=True) for idx in g]
        best_idx = g[np.argmax(kurt_vals)]
        for idx in g:
            if idx != best_idx:
                orig_idx = original_indices[idx]
                iscell_updated[orig_idx, 0] = 0
                n_removed += 1

np.save(iscell_path, iscell_updated)

n_before = int(np.sum(iscell[:, 0] == 1))
n_after = int(np.sum(iscell_updated[:, 0] == 1))

print(f"\niscell updated:")
print(f"  Before: {n_before} cells")
print(f"  After:  {n_after} cells  (= {n_merged} independent axons)")
print(f"  Removed (redundant members of merged groups): {n_removed}")
print(f"\nSaved to: {iscell_path}")
print(f"Open suite2p GUI to review the surviving ROIs.")